# Powered at 80% to detect an effect you already believe

The power calculation gets done, the number comes back 0.8, the experiment is approved. Two
questions were skipped. The first: is the effect it is powered for one anybody doubts? If the
fitted model already puts 96% of its mass above the minimum detectable effect, the experiment
can only confirm — it is not a test, it is a formality with a budget.

The second: was the sd in the denominator the sd of independent observations? Randomize
twenty-four regions of forty stores and the arithmetic says 960 units. With any within-region
correlation at all it is not 960, it is closer to 300, and the experiment is underpowered by
the time it starts.

Every number here comes from one normal-model formula. A two-arm difference in means with
`n` units in total, outcome sd `sd` and treated share `allocation` has standard error
`se = sd / sqrt(n · p · (1 − p))`; the power to detect `δ` at level `α` is then

    two-sided:  Φ(|δ|/se − z_{1−α/2}) + Φ(−|δ|/se − z_{1−α/2})

`mde` and `sample_size` are *exact* inversions of that formula (not the textbook
`(z_α + z_β)·se` approximation), so `power(mde(...))` reproduces the target. Every result is
a `Spec` that carries `alpha`, `power` and `two_sided` with it.

In [ ]:
import numpy as np

from axiom.core import D, Posterior, Unit, Unsupported
from axiom.design import (
    MDE, AnchoredEffect, Assignment, ClusterDesign, CostPerOutcomeInterval, CostPerOutcomePower,
    HoldoutTradeoff, MatchMetric, PowerCurve, PowerResult, SampleSize, anchor_draws, anchor_effect,
    cluster_mde, cluster_power, clusters_needed, coefficient_mde, coefficient_power,
    coefficient_sample_size, cost_per_outcome_interval, cost_per_outcome_power, design_effect,
    difference_se, effective_sample_size, holdout_tradeoff, match_clusters,
    max_detectable_cost_per_outcome, mde, power, power_curve, power_from_se, sample_size,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, CRITICAL, ORANGE, annotate, caption, compare, curve_band, density, intervals, lines, mark_x, mark_y, shade

enable();  # every axiom result renders itself from here on

In [ ]:
se = difference_se(200, sd=2.0, allocation=0.5)
pr: PowerResult = power(200, effect=0.6, sd=2.0)
print(f"se = {se:.4f}  power = {pr.power:.3f}  (alpha={pr.alpha}, two_sided={pr.two_sided})")
print("one-sided, 40% treated:", round(power(200, 0.6, 2.0, two_sided=False, allocation=0.4).power, 3))

In [ ]:
m: MDE = mde(200, sd=2.0, power=0.8)
print(f"MDE at n=200: {m.effect:.4f}   check power(mde) = {power(200, m.effect, 2.0).power:.6f}")
ss = sample_size(effect=0.6, sd=2.0, power=0.8)
assert isinstance(ss, SampleSize)
print(f"n for effect 0.6: {ss.n} ({ss.n_treated} treated / {ss.n_control} control), achieved power {ss.power:.4f}")
print("zero effect ->", type(sample_size(0.0, 2.0)).__name__)

In [ ]:
curve: PowerCurve = power_curve(200, 2.0, effects=np.linspace(0.0, 1.2, 7))
table(
    [[f"{e:.2f}", f"{p:.3f}"] for e, p in zip(curve.effects, curve.powers)],
    headers=("effect", "power"),
)
print("interpolated at 0.5:", round(curve.power_at(0.5), 3))

In [ ]:
fine = np.linspace(0.0, 1.2, 120)
fig = curve_band(
    fine, [power(200, float(e), 2.0).power for e in fine],
    label="power",
    title="What 'powered at 80%' is a statement about",
    subtitle="two arms, n = 200, outcome sd 2.0, α = 0.05 two-sided",
    x_title="true effect", y_title="probability of rejecting",
)
mark_y(fig, 0.8, text="80%")
mark_x(fig, m.effect, text=f"MDE = {m.effect:.2f}", color=ORANGE)
shade(fig, 0.0, m.effect, text="invisible to this experiment", color=CRITICAL, alpha=0.07)
caption(fig, "Everything left of the orange line is an effect this experiment will usually "
             "miss. Whether that matters depends entirely on whether such effects are "
             "plausible — which is a question about the prior, not about the design.")

## Regression coefficients

A coefficient has whatever design standard error the design gives it. The `coefficient_*`
functions take that `se` directly; for sample size they assume the design is replicated, so
`se(m) = se · sqrt(n / m)`.

In [ ]:
print("power_from_se:", round(power_from_se(0.3, 0.1).power, 3))
print("coefficient_power:", round(coefficient_power(0.3, se=0.1).power, 3))
print("coefficient_mde:", round(coefficient_mde(se=0.1, power=0.9).effect, 4))
css = coefficient_sample_size(effect=0.2, se=0.1, n=500)
assert isinstance(css, SampleSize)
print(f"coefficient_sample_size: n={css.n}, se at n={css.se:.4f}, power={css.power:.4f}")

## Cluster-randomized designs

Randomizing `k` clusters of `m` individuals is worth fewer than `k·m` independent observations
when outcomes within a cluster are correlated. With intra-cluster correlation `icc` the
inflation is the design effect `DE = 1 + (m − 1)·icc`; every cluster function reduces to
`power` with `n = k·m` and `sd·sqrt(DE)`. There is no second power formula.

In [ ]:
region = Unit(name="region", dimension=D.entity, kind="cluster")
cd = ClusterDesign(unit=region, n_clusters=24, cluster_size=40, icc=0.05, allocation=0.5)
print("design effect:", design_effect(40, 0.05), "| effective n:", round(effective_sample_size(24, 40, 0.05), 1))
print("cluster power for effect 0.5:", round(cluster_power(cd, effect=0.5, sd=2.0).power, 3))
print("cluster MDE:", round(cluster_mde(cd, sd=2.0).effect, 4))
cn = clusters_needed(effect=0.5, sd=2.0, cluster_size=40, icc=0.05)
assert isinstance(cn, SampleSize)
print(f"clusters needed: {cn.n} ({cn.n_treated} treated), achieved power {cn.power:.3f}")

In [ ]:
sizes = np.arange(4, 61, 2)
by_icc = {
    f"icc = {icc}": [cluster_power(ClusterDesign(unit=region, n_clusters=int(k), cluster_size=40, icc=icc), effect=0.5, sd=2.0).power
                     for k in sizes]
    for icc in (0.0, 0.02, 0.05)
}
fig = lines(
    sizes, by_icc,
    colors=(BLUE, ORANGE, AQUA),
    title="Forty stores in a region are not forty observations",
    subtitle="power to detect an effect of 0.5, clusters of 40, against the number of clusters randomized",
    x_title="clusters randomized", y_title="power",
)
mark_y(fig, 0.8, text="80%")
caption(fig, f"At icc = 0.05 the design effect is {design_effect(40, 0.05):.1f}: forty stores "
             f"per region count for {40 / design_effect(40, 0.05):.1f}. The top curve is the "
             f"calculation that gets done when nobody asks about correlation within a region.")

In [ ]:
rng = np.random.default_rng(0)
pre = rng.normal(10.0, 2.0, size=(9, 1)) + rng.normal(0.0, 0.3, size=(9, 6))
metric: MatchMetric = "trajectory"
asg: Assignment = match_clusters(pre, labels=[f"r{i}" for i in range(9)], unit=region, metric=metric, seed=1)
print("pairs:", asg.pairs, "| unpaired:", asg.unpaired)
print("treated:", asg.treated, "control:", asg.control)
print(f"pre-period SMD {asg.pre_smd:.3f}, mean pair distance {asg.mean_pair_distance:.3f}")

In [ ]:
ht: HoldoutTradeoff = holdout_tradeoff(cd, sd=2.0, fractions=(0.1, 0.2, 0.3, 0.4, 0.5))
table(
    [[f"{f:.1f}", f"{m_:.3f}", f"{r:.2f}x"] for f, m_, r in zip(ht.fractions, ht.mdes, ht.relative_mde)],
    headers=("holdout", "MDE", "against the best"),
)
print("best fraction:", ht.best_fraction)

In [ ]:
fine_ht = holdout_tradeoff(cd, sd=2.0, fractions=tuple(np.round(np.linspace(0.05, 0.5, 10), 3)))
fig = curve_band(
    fine_ht.fractions, fine_ht.mdes,
    label="MDE",
    title="How much of the map do you dark?",
    subtitle="minimum detectable effect against the share of clusters held out",
    x_title="holdout fraction", y_title="MDE",
)
mark_x(fig, fine_ht.best_fraction, text=f"best: {fine_ht.best_fraction:.0%}")
caption(fig, "The curve is steep on the left and nearly flat past 30%: going from a 10% "
             "holdout to 30% buys a third off the MDE, and going from 30% to 50% buys another "
             "9% while doubling the outcome you gave up to get it. That trade is the decision, "
             "and it is now a number rather than a habit.")

## Anchoring the MDE to a posterior

If a fitted model already puts most of its mass above the conventional MDE, the experiment is
powered to confirm a belief, not to test it. `anchor_effect` reports the posterior probability
of exceeding the MDE and the `1 − credence` quantile — the effect the model *doubts*, which is
what a test should be powered for.

In [ ]:
draws = rng.normal(0.9, 0.25, size=(2, 500))
posterior = Posterior({"beta": draws})
ae = anchor_effect(posterior, "beta", mde=0.5, credence=0.9)
assert isinstance(ae, AnchoredEffect)
print(f"P(beta > 0.5) = {ae.probability_exceeds_mde:.3f} (gaussian {ae.probability_exceeds_mde_gaussian:.3f})")
print(f"anchored effect (10% quantile) = {ae.anchored_effect:.3f}; already believed: {ae.already_believed}")
print("from flat draws:", round(anchor_draws(draws.ravel(), "beta", 0.5).anchored_effect, 3))
print("missing parameter ->", anchor_effect(posterior, "gamma", 0.5).reason)

In [ ]:
fig = density(
    {"what the model already believes": draws.ravel()},
    colors=(BLUE,),
    title="The experiment at the top of this notebook",
    subtitle="posterior for the effect, against the MDE the design was powered for",
    x_title="effect",
)
mark_x(fig, 0.5, text="MDE = 0.5", color=ORANGE)
mark_x(fig, ae.anchored_effect, text=f"anchored effect = {ae.anchored_effect:.2f}", color=CRITICAL, right=True)
shade(fig, 0.5, float(draws.max()), text=f"{ae.probability_exceeds_mde:.0%} of the mass", color=ORANGE, alpha=0.08)
caption(fig, "Powering for 0.5 buys a test the model is already {:.0%} sure will pass. The "
             "effect worth designing against is the one the model doubts — the 10% quantile "
             "at {:.2f} — and it needs a bigger experiment.".format(ae.probability_exceeds_mde, ae.anchored_effect))

## Cost per outcome unit

`cost / effect` has a noisy denominator, so its interval is asymmetric. The Fieller interval
inverts the effect's Wald interval, `cost / (effect ± z·se)`; when that interval reaches zero
the ratio is unbounded above and the result says so rather than printing a finite number. The
naive delta-method interval is reported alongside for comparison.

In [ ]:
ci: CostPerOutcomeInterval = cost_per_outcome_interval(cost=1000.0, effect=25.0, effect_se=6.0)
print(f"estimate {ci.estimate:.2f}  fieller [{ci.lower:.2f}, {ci.upper:.2f}]  status={ci.status}")
print("naive:", ci.naive_interval)
wide = cost_per_outcome_interval(1000.0, 25.0, 15.0)
print("noisy effect ->", wide.status, "upper =", wide.upper)
cp: CostPerOutcomePower = cost_per_outcome_power(1000.0, true_effect=25.0, effect_se=6.0, threshold=60.0)
print(f"P(upper bound < 60) = {cp.power:.3f}  (effect required {cp.effect_required:.2f})")
print("largest bounded cost per outcome at the MDE:", round(max_detectable_cost_per_outcome(1000.0, m.effect), 2))

In [ ]:
rows = [
    ("Fieller (effect se 6)", ci.estimate, ci.lower, ci.upper),
    ("naive delta method (se 6)", ci.estimate, ci.naive_interval.lower, ci.naive_interval.upper),
    ("naive delta method (se 15)", wide.estimate, wide.naive_interval.lower, wide.naive_interval.upper),
]
fig = intervals(
    rows,
    highlight="naive delta method (se 15)",
    title="Cost per outcome has a noisy denominator",
    subtitle=f"cost 1000 over an effect of 25 — the Fieller interval against the symmetric approximation",
    x_title="cost per outcome unit",
)
caption(fig, f"At an effect se of 15 the Fieller interval is unbounded above ({wide.status}) "
             f"because the effect's own interval reaches zero — a cost per outcome of infinity "
             f"is on the table. The bottom row is the symmetric interval that would have been "
             f"reported instead, and it ends at a finite number that does not exist.")

## What this bought you

Power that is an exact inversion rather than the textbook approximation, cluster designs that
count effective observations rather than rows, a holdout fraction chosen off a curve, an MDE
checked against what the model already believes, and a cost-per-outcome interval that says
"unbounded" when it is unbounded.

`02-eig-and-evoi.ipynb` prices the same designs in information and in money.